# Bengaluru Climate Risk Scanner
## Report maps: flood (Phase 2) and heat (Phase 3)

Makes clean, print-ready maps from the saved outputs of Notebooks 02 and 03. Each map has a title, a small legend, a north arrow, a scale bar, a neat frame and a source line, and is saved as a 300 dpi PNG in `MyDrive/bengaluru-climate-risk/maps/`.

It only reads files from Drive, so you can run it any time after Notebooks 02 and 03, even after a restart.

**Colour logic.** Every flood map uses one purple ramp and every heat map one orange-red ramp: light = safe, dark = risky. A single-hue ramp reads correctly for colour-blind readers and in greyscale print. Lakes are always light blue.

## 1. Connect Drive and load the configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
with open(f'{PROJECT_DIR}/config.json') as f:
    CONFIG = json.load(f)
OUT = CONFIG['paths']['processed']
GPKG = CONFIG['paths']['gpkg']
MAPS = f'{PROJECT_DIR}/maps'
os.makedirs(MAPS, exist_ok=True)

AUTHOR = 'Gowri'          # shown in the footer of every map; change to your full name
print('Maps will be saved to', MAPS)

## 2. Load the saved layers

In [ ]:
import numpy as np
import geopandas as gpd
import rasterio

def read(name, band=1):
    with rasterio.open(f'{OUT}/{name}') as src:
        a = src.read(band).astype('float32')
        if src.nodata is not None:
            a[a == src.nodata] = np.nan
        return a, src.transform, src.crs

fri, transform, crs = read('flood_risk.tif')
flood_class, _, _ = read('flood_class.tif')
F = {name: read('flood_factor_values.tif', i)[0]
     for i, name in enumerate(['hand', 'slope', 'dist_lake', 'dist_drain', 'twi', 'builtup'], start=1)}

heat, _, _ = read('heat_risk.tif')
heat_class, _, _ = read('heat_class.tif')
lst, _, _ = read('heat_values.tif', 1)
ndvi, _, _ = read('heat_values.tif', 2)

boundary = gpd.read_file(GPKG, layer='bbmp_boundary').to_crs(crs)
lakes = gpd.read_file(GPKG, layer='lakes').to_crs(crs)
lakes_city = gpd.clip(lakes, boundary.buffer(3000))

ny, nx = fri.shape
res = transform.a
EXTENT = [transform.c, transform.c + nx * res, transform.f - ny * res, transform.f]
city_mask = ~np.isnan(fri) | (flood_class == 0)
outside = lambda a: np.where(city_mask, a, np.nan)

FM, HM = CONFIG['flood_model'], CONFIG['heat_model']
print('Loaded. Grid', nx, 'x', ny, 'at', res, 'm')

## 3. Map style and layout helpers
`map_frame()` draws everything that's the same on every map: page border, title, subtitle, map frame, city outline, lakes, north arrow, scale bar and footer. `classed_map()` is for maps with a few classes (legend of colour boxes); `continuous_map()` is for smooth values (small colour bar).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.patches import Patch, Polygon, Rectangle, FancyBboxPatch

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})
INK, MUTED, FRAME = '#1f2328', '#6e7781', '#8c959f'
LAKE = '#a6cee3'

FLOOD_5 = ['#efedf5', '#cbc9e2', '#9e9ac8', '#756bb1', '#54278f']   # purple: safe -> very risky
HEAT_5  = ['#fef0d9', '#fdcc8a', '#fc8d59', '#e34a33', '#b30000']   # orange-red
GREEN_5 = ['#edf8e9', '#bae4b3', '#74c476', '#31a354', '#006d2c']
FLOOD_3 = [FLOOD_5[1], FLOOD_5[2], FLOOD_5[4]]
HEAT_3  = [HEAT_5[1], HEAT_5[2], HEAT_5[4]]

SOURCES_FLOOD = 'Data: FABDEM (Copernicus GLO-30), ESA WorldCover 2021, OpenStreetMap contributors'
SOURCES_HEAT  = 'Data: Landsat 8/9 Collection 2 Level 2 (USGS), Mar–May 2022–2024, OpenStreetMap contributors'

xmin, ymin, xmax, ymax = boundary.total_bounds
PAD = 1500
VIEW = (xmin - PAD, xmax + PAD, ymin - PAD, ymax + PAD)

def north_arrow(ax, x=0.93, y=0.86, h=0.075, w=0.022):
    tip, base_l, base_r, notch = (x, y + h), (x - w, y), (x + w, y), (x, y + h * 0.28)
    ax.add_patch(Polygon([tip, base_l, notch], closed=True, transform=ax.transAxes, fc=INK, ec=INK, lw=0.8, zorder=10))
    ax.add_patch(Polygon([tip, notch, base_r], closed=True, transform=ax.transAxes, fc='white', ec=INK, lw=0.8, zorder=10))
    ax.text(x, y + h + 0.012, 'N', transform=ax.transAxes, ha='center', va='bottom',
            fontsize=10, fontweight='bold', color=INK, zorder=10)

def scale_bar(ax):
    x0, x1, y0, y1 = VIEW
    width_km = (x1 - x0) / 1000
    length = next(L for L in (20, 10, 5, 2, 1) if L <= width_km / 4)
    seg = length * 1000 / 4
    bx = x1 - (x1 - x0) * 0.05 - length * 1000
    by = y0 + (y1 - y0) * 0.045
    hgt = (y1 - y0) * 0.008
    ax.add_patch(Rectangle((bx - seg * 0.35, by - hgt * 3.2), length * 1000 + seg * 1.3, hgt * 7.6,
                           fc='white', ec='none', alpha=0.85, zorder=9))
    for i in range(4):
        ax.add_patch(Rectangle((bx + i * seg, by), seg, hgt, fc=INK if i % 2 == 0 else 'white',
                               ec=INK, lw=0.6, zorder=10))
    for frac, label in ((0, '0'), (0.5, f'{length / 2:g}'), (1, f'{length:g} km')):
        ax.text(bx + frac * length * 1000, by + hgt * 1.6, label, ha='center', va='bottom',
                fontsize=7.5, color=INK, zorder=10)

def map_frame(title, subtitle, sources):
    fig = plt.figure(figsize=(8.27, 9.6), dpi=110, facecolor='white')
    fig.patches.append(Rectangle((0.012, 0.012), 0.976, 0.976, transform=fig.transFigure,
                                 fill=False, ec=INK, lw=1.4))
    fig.patches.append(Rectangle((0.02, 0.02), 0.96, 0.96, transform=fig.transFigure,
                                 fill=False, ec=FRAME, lw=0.5))
    fig.text(0.06, 0.945, title, fontsize=16, fontweight='bold', color=INK, va='top')
    fig.text(0.06, 0.905, subtitle, fontsize=10, color=MUTED, va='top')

    ax = fig.add_axes([0.06, 0.1, 0.88, 0.78])
    ax.set_xlim(VIEW[0], VIEW[1]); ax.set_ylim(VIEW[2], VIEW[3]); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor(FRAME); s.set_linewidth(0.8)
    ax.set_facecolor('#f6f8fa')

    fig.text(0.06, 0.07, sources, fontsize=7, color=MUTED)
    fig.text(0.06, 0.05, 'Projection: WGS 84 / UTM zone 43N (EPSG:32643)  ·  30 m grid', fontsize=7, color=MUTED)
    fig.text(0.94, 0.05, f'Bengaluru Climate Risk Scanner  ·  {AUTHOR}', fontsize=7, color=MUTED, ha='right')
    return fig, ax

def finish(fig, ax, name, show_lakes=True):
    if show_lakes and len(lakes_city):
        lakes_city.plot(ax=ax, color=LAKE, edgecolor='none', zorder=4)
    boundary.boundary.plot(ax=ax, color=INK, linewidth=0.9, zorder=5)
    north_arrow(ax); scale_bar(ax)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xlim(VIEW[0], VIEW[1]); ax.set_ylim(VIEW[2], VIEW[3])
    path = f'{MAPS}/{name}.png'
    fig.savefig(path, dpi=300, facecolor='white')
    plt.show()
    print('Saved', path)

def legend_box(ax, handles, title):
    leg = ax.legend(handles=handles, title=title, loc='lower left', fontsize=7.5, title_fontsize=8,
                    frameon=True, framealpha=0.92, edgecolor=FRAME, fancybox=False,
                    handlelength=1.4, handleheight=1.0, borderpad=0.7, labelspacing=0.35)
    leg.get_title().set_fontweight('bold'); leg.get_title().set_color(INK)
    leg.set_zorder(11)

def classed_map(name, arr, colors, labels, title, subtitle, legend_title, sources, lakes_label=True):
    fig, ax = map_frame(title, subtitle, sources)
    k = len(colors)
    ax.imshow(outside(arr), cmap=ListedColormap(colors), norm=BoundaryNorm(np.arange(0.5, k + 1), k),
              extent=EXTENT, interpolation='nearest', zorder=2)
    handles = [Patch(fc=c, ec=FRAME, lw=0.4, label=l) for c, l in zip(colors[::-1], labels[::-1])]
    if lakes_label:
        handles.append(Patch(fc=LAKE, ec='none', label='Lake / water body'))
    legend_box(ax, handles, legend_title)
    finish(fig, ax, name)

def continuous_map(name, arr, colors, vmin, vmax, title, subtitle, cbar_label, sources, ticks=None):
    fig, ax = map_frame(title, subtitle, sources)
    cmap = LinearSegmentedColormap.from_list(name, colors)
    im = ax.imshow(outside(arr), cmap=cmap, vmin=vmin, vmax=vmax, extent=EXTENT, interpolation='nearest', zorder=2)
    ax.add_patch(FancyBboxPatch((0.025, 0.025), 0.34, 0.115, boxstyle='square,pad=0', transform=ax.transAxes,
                                fc='white', ec=FRAME, lw=0.6, alpha=0.92, zorder=10))
    cax = ax.inset_axes([0.05, 0.055, 0.29, 0.022], zorder=11)
    cb = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=ticks)
    cb.outline.set_linewidth(0.5); cb.ax.tick_params(labelsize=7, length=2, width=0.5, colors=INK)
    ax.text(0.05, 0.1, cbar_label, transform=ax.transAxes, fontsize=8, fontweight='bold', color=INK, zorder=11)
    finish(fig, ax, name)

def ranges(breaks, unit, fmt='{:g}'):
    """Legend labels for classes split by breaks, low to high: '< 1.5 m', '1.5–3 m', ..., '≥ 10 m'."""
    b = [fmt.format(x) for x in breaks]
    return [f'< {b[0]} {unit}'] + [f'{b[i]}–{b[i + 1]} {unit}' for i in range(len(b) - 1)] + [f'≥ {b[-1]} {unit}']

## 4. Phase 2: flood factor maps
Each factor is drawn with the same classes used for scoring, so the map explains the score: the darkest class is the one that scores 5 (most flood-prone).

In [ ]:
def score_classes(values, breaks, low_is_risky):
    idx = np.digitize(np.nan_to_num(values, nan=-1e9 if not low_is_risky else 1e9), breaks)  # 0..4
    score = (5 - idx) if low_is_risky else (1 + idx)
    return np.where(np.isnan(values), np.nan, score).astype('float32')

factor_maps = [
    ('flood_factor_hand', 'hand', FM['hand_breaks_m'], True, 'm',
     'Height above nearest drainage', 'How far each place sits above the stream, drain or lake it drains to', 'Height (HAND)'),
    ('flood_factor_slope', 'slope', FM['slope_breaks_deg'], True, '°',
     'Slope', 'Flat ground drains slowly and holds water', 'Slope'),
    ('flood_factor_lakes', 'dist_lake', FM['lake_breaks_m'], True, 'm',
     'Distance to lakes', 'Bengaluru floods start at overflowing lakes', 'Distance'),
    ('flood_factor_drains', 'dist_drain', FM['drain_breaks_m'], True, 'm',
     'Distance to drains and streams', 'Overflowing stormwater drains spread water', 'Distance'),
    ('flood_factor_twi', 'twi', FM['twi_breaks'], False, '',
     'Topographic wetness index', 'Where runoff gathers: large drainage area on flat ground', 'TWI'),
    ('flood_factor_builtup', 'builtup', FM['builtup_breaks'], False, '',
     'Built-up share', 'Concrete and roofs stop rain soaking in', 'Share of 30 m cell'),
]

for name, key, breaks, low_is_risky, unit, title, subtitle, leg_title in factor_maps:
    labels = ranges(breaks, unit, fmt='{:.1f}' if key == 'twi' else '{:g}')
    if key == 'builtup':
        labels = ranges([int(b * 100) for b in breaks], '%')
    sc = score_classes(F[key], breaks, low_is_risky)
    # colours run with the score (1 light ... 5 dark); labels run with the value
    labels_by_score = labels[::-1] if low_is_risky else labels
    labels_by_score = [f'{l}  (score {s})' for s, l in zip(range(1, 6), labels_by_score)]
    weight = FM['weights_used'][key]
    classed_map(name, sc, FLOOD_5, labels_by_score, title, f'{subtitle}  ·  weight {weight:.2f}',
                leg_title, SOURCES_FLOOD)

## 5. Phase 2: flood risk index and flood classes

In [ ]:
continuous_map('flood_risk_index', fri, FLOOD_5, 1, 5,
               'Flood risk index', 'Weighted overlay of six factors · 1 = safe, 5 = very flood-prone',
               'Flood risk index', SOURCES_FLOOD, ticks=[1, 2, 3, 4, 5])

areas = {c: np.nansum(flood_class == c) * res * res / 1e6 for c in (1, 2, 3)}
tot = sum(areas.values())
lab = [f'{l}  ·  {areas[c]:.0f} km² ({100 * areas[c] / tot:.0f}%)' for c, l in zip((1, 2, 3), ('Low', 'Medium', 'High'))]
classed_map('flood_classes', np.where(flood_class == 0, np.nan, flood_class), FLOOD_3, lab,
            'Flood susceptibility', f'Low / Medium / High by FRI percentiles (breaks {FM["class_breaks_fri"][0]} and {FM["class_breaks_fri"][1]})',
            'Flood risk', SOURCES_FLOOD)

## 6. Phase 3: summer temperature, greenness, heat index and heat classes

In [ ]:
lo, hi = np.nanpercentile(lst, [2, 98])
lo, hi = np.floor(lo), np.ceil(hi)
continuous_map('heat_lst', lst, HEAT_5, lo, hi,
               'Summer land surface temperature', 'Median of clear Landsat 8/9 scenes, March–May 2022–2024',
               'Surface temperature (°C)', SOURCES_HEAT, ticks=np.linspace(lo, hi, 5).round(0))

continuous_map('heat_ndvi', ndvi, GREEN_5, 0, 0.6,
               'Vegetation (NDVI)', 'Greener surfaces stay cooler through shade and evaporation',
               'NDVI', SOURCES_HEAT, ticks=[0, 0.2, 0.4, 0.6])

continuous_map('heat_index', heat, HEAT_5, 0, 1,
               'Heat index', '0.7 × normalised temperature + 0.3 × (1 − normalised NDVI)',
               'Heat index', SOURCES_HEAT, ticks=[0, 0.25, 0.5, 0.75, 1])

areas = {c: np.nansum(heat_class == c) * res * res / 1e6 for c in (1, 2, 3)}
tot = sum(areas.values())
mean_t = {c: np.nanmean(lst[heat_class == c]) for c in (1, 2, 3)}
lab = [f'{l}  ·  {areas[c]:.0f} km² · {mean_t[c]:.1f} °C' for c, l in zip((1, 2, 3), ('Low', 'Medium', 'High'))]
uhi = HM.get('surface_uhi_c')
sub = 'Low / Medium / High by heat-index percentiles' + (f'  ·  urban heat island {uhi:+.1f} °C' if uhi is not None else '')
classed_map('heat_classes', np.where(heat_class == 0, np.nan, heat_class), HEAT_3, lab,
            'Extreme-heat zones', sub, 'Heat risk  ·  area · mean LST', SOURCES_HEAT)

## 7. Files saved

In [ ]:
for f in sorted(os.listdir(MAPS)):
    print(f'  {f:<28} {os.path.getsize(f"{MAPS}/{f}") / 1e6:5.1f} MB')